In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch


X_train_tensor = torch.tensor(X_train).float()
X_test_tensor = torch.tensor(X_test).float()
y_train_tensor = torch.tensor(y_train).long()
y_test_tensor = torch.tensor(y_test).long()




In [ ]:
# 2. Create TensorDataset objects
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
# Create TensorDatasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)


In [ ]:
# 3. Create DataLoaders
# Create DataLoaders with a batch size of 32
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [ ]:
# 4. Print shape of one batch

# Get a single batch from the train loader
data_iter = iter(train_loader)
images, labels = next(data_iter)

# Print the shape of the images and labels
print(f"Image batch shape: {images.shape}")
print(f"Labels batch shape: {labels.shape}")

In [ ]:
# 5. Display sample images
import matplotlib.pyplot as plt

# Display the first 5 images from the batch
figure = plt.figure(figsize=(10, 4))
num_of_images = 5

for index in range(num_of_images):
    plt.subplot(1, num_of_images, index + 1)

    # Grab the image tensor
    img = images[index]


    if img.ndim == 3 and img.shape[0] in [1, 3]:
        img = img.permute(1, 2, 0)
        # If grayscale (1, H, W), squeeze to (H, W)
        if img.shape[2] == 1:
            img = img.squeeze()

    plt.imshow(img, cmap='gray') # Remove cmap='gray' if images are RGB
    plt.title(f"Label: {labels[index].item()}")
    plt.axis('off')

plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Generate dummy data (1000 samples, 10 features)
X = torch.randn(1000, 10)
y = torch.randint(0, 2, (1000,)).float().unsqueeze(1) # Binary targets

# Split and create DataLoaders
train_ds = TensorDataset(X[:800], y[:800])
val_ds = TensorDataset(X[800:], y[800:])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNet, self).__init__()
        # Defining the 4 Linear Layers
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.layer2 = nn.Linear(hidden_size, hidden_size)
        self.layer3 = nn.Linear(hidden_size, hidden_size)
        self.layer4 = nn.Linear(hidden_size, output_size)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.relu(self.layer3(x))
        x = self.layer4(x) # No activation on output (using logits)
        return x

In [ ]:
# Task 3: Write your validation loop here:
def train_step(model, dataloader, loss_fn, optimizer, device):
    model.train() # Enable training mode
    running_loss = 0.0

    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = loss_fn(outputs, targets)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(dataloader)

In [ ]:
# Task 4: Define device, model, loss, optimizer:

def validate_step(model, dataloader, loss_fn, device):
    model.eval() # Enable evaluation mode
    running_loss = 0.0

    with torch.inference_mode(): # turn off gradient tracking
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)

            outputs = model(inputs)
            loss = loss_fn(outputs, targets)

            running_loss += loss.item()

    return running_loss / len(dataloader)



In [ ]:
# Task 5: Start training for 20 epochs:
# 1. Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 2. Model
INPUT_SIZE = 10
HIDDEN_SIZE = 64
OUTPUT_SIZE = 1

model = SimpleNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE).to(device)

# 3. Loss Function (BCEWithLogitsLoss combines Sigmoid + BCELoss)
loss_fn = nn.BCEWithLogitsLoss()

# 4. Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
EPOCHS = 20

print(f"Starting training on {device}...")

for epoch in range(EPOCHS):
    # Run loops
    train_loss = train_step(model, train_loader, loss_fn, optimizer, device)
    val_loss = validate_step(model, val_loader, loss_fn, device)

    # Track results
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

print("Training Complete.")

In [ ]:
# Task 1: Write your code here:
train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    train_loss = train_step(model, train_loader, loss_fn, optimizer, device)
    val_loss = validate_step(model, val_loader, loss_fn, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

In [ ]:
# Task 2 (Bonus): Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
dummy_images = torch.randn(16, 1, 28, 28)
dummy_ages = torch.randint(18, 90, (16,)).float()
predicted_ages = dummy_ages + torch.randn(16) * 5

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
axes = axes.flatten()

for i, ax in enumerate(axes):
    img = dummy_images[i].squeeze()
    actual = dummy_ages[i].item()
    pred = predicted_ages[i].item()

    ax.imshow(img, cmap='gray')
    ax.set_title(f"Actual: {actual:.1f}\nPred: {pred:.1f}")
    ax.axis('off')

plt.tight_layout()
plt.show()